# <b>ArUco Marker Detection</b>

In [ ]:
%pip install opencv-contrib-python

In [ ]:
pip show opencv-contrib-python

In [ ]:
from picamera2 import Picamera2
import cv2
import ipywidgets as widgets
from IPython.display import display, clear_output
import time

# ArUco 설정 (DICT_4X4_50, ID 0~15 사용)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

video_widget = widgets.Image(
    format='jpeg',
    layout=widgets.Layout(width='320px', height='180px')
)
text_widget = widgets.Label(value="Scanning ArUco markers...")

display(widgets.VBox([video_widget, text_widget]))

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

try:
    while True:
        frame = picam2.capture_array()
        frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

        # ArUco 마커 검출
        corners, ids, rejected = detector.detectMarkers(frame)

        marker_ids = []
        if ids is not None:
            marker_ids = [int(i) for i in ids.flatten() if 0 <= int(i) <= 15]

        if marker_ids:
            text_widget.value = "ArUco ID: " + ", ".join(map(str, marker_ids))
        else:
            text_widget.value = "Scanning ArUco markers..."

        video_widget.value = convert_to_bytes(frame)

        clear_output(wait=True)
        display(widgets.VBox([video_widget, text_widget]))

        # 이미지 업데이트를 5 FPS로 제한
        time.sleep(0.2)

except KeyboardInterrupt:
    pass

finally:
    picam2.stop()

# <b>ArUco Marker Bbox</b>

In [ ]:
from picamera2 import Picamera2
import cv2
from tiki.mini import TikiMini
import ipywidgets as widgets
from IPython.display import display, clear_output
import numpy as np
import time

# TikiMini 인스턴스 생성
tiki = TikiMini()

# ArUco 설정 (DICT_4X4_50, ID 0~15 사용)
aruco_dict = cv2.aruco.getPredefinedDictionary(cv2.aruco.DICT_4X4_50)
parameters = cv2.aruco.DetectorParameters()
detector = cv2.aruco.ArucoDetector(aruco_dict, parameters)

# 비디오 캡처 객체 생성
picam2 = Picamera2()
camera_config = picam2.create_preview_configuration(
    main={"size": (320, 180), "format": "BGR888"}
)
picam2.configure(camera_config)
picam2.start()

marker_ids = []

def convert_to_bytes(image):
    _, buffer = cv2.imencode('.jpg', image)
    return buffer.tobytes()

def process_frame(frame):
    """ArUco 마커를 검출하고 ID 0~15의 바운딩 박스를 표시합니다."""
    global marker_ids

    corners, ids, rejected = detector.detectMarkers(frame)
    marker_ids = []

    if ids is not None:
        valid_corners = []
        valid_ids = []

        for corner, marker_id in zip(corners, ids.flatten()):
            marker_id = int(marker_id)

            if 0 <= marker_id <= 15:
                marker_ids.append(marker_id)
                valid_corners.append(corner)
                valid_ids.append([marker_id])

        if valid_ids:
            cv2.aruco.drawDetectedMarkers(
                frame,
                valid_corners,
                np.array(valid_ids, dtype=np.int32)
            )

    return frame

def update_display():
    frame = picam2.capture_array()
    frame = cv2.cvtColor(frame, cv2.COLOR_RGB2BGR)

    frame = process_frame(frame)

    # 비디오 위젯 업데이트
    video_widget.value = convert_to_bytes(frame)

    # ArUco ID 출력 및 TIKI OLED 로그
    if marker_ids:
        marker_text = ", ".join(map(str, marker_ids))
        text_widget.value = f"ArUco ID: {marker_text}"
        tiki.log(f"ArUco ID: {marker_text}")
    else:
        text_widget.value = "Scanning ArUco markers..."

    clear_output(wait=True)
    display(widgets.VBox([text_widget, video_widget]))

video_widget = widgets.Image(
    format='jpeg',
    layout=widgets.Layout(width='320px', height='180px')
)
text_widget = widgets.Label(value="Scanning ArUco markers...")

display(widgets.VBox([text_widget, video_widget]))

try:
    while True:
        update_display()

        # 이미지 업데이트를 5 FPS로 제한
        time.sleep(0.2)

except KeyboardInterrupt:
    pass

finally:
    picam2.stop()